In [0]:
%pip install vaderSentiment

In [0]:
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [0]:
df_bronze = spark.table("us_economy_db.bronze_bsky_posts")

In [0]:
financial_keywords = r"(?i)\b(price|market|oz|ounce|bullion|invest|inflation|fed|economy|trading|xau|spot|rally|rate|dollar|bank|stock|etf|reserve|mining|commodity)\b"

df_filtered = (
    df_bronze
    .dropDuplicates(["post_uri"])
    .filter(
        (F.col("search_topic") != "gold")
        | (F.col("raw_text").rlike(financial_keywords))
    )
)

In [0]:
df_cleaned = (
    df_filtered.withColumn(
        "clean_text",
        F.regexp_replace(
            F.col("raw_text"), r"http\S+|www\S+|https\S+|youtube\.com\S+", ""
        ),
    )
    .withColumn(
        "clean_text",
        F.regexp_replace(F.col("clean_text"), r"@[A-Za-z0-9_.-]+", ""),
    ) 
    .withColumn(
        "clean_text", F.regexp_replace(F.col("clean_text"), r"[\ufffc\r\n]+", " ")
    )
    .withColumn(
        "clean_text", F.trim(F.regexp_replace(F.col("clean_text"), r"\s+", " "))
    )
    .withColumn("post_timestamp", F.to_timestamp(F.col("created_at")))
    .withColumn("post_date", F.to_date(F.col("post_timestamp")))
    .withColumn("post_hour", F.hour(F.col("post_timestamp")))
    .filter(F.length(F.col("clean_text")) > 15)
    .filter(F.col("post_date").isNotNull())
)

In [0]:
@F.pandas_udf(FloatType())
def fast_sentiment_udf(texts: pd.Series) -> pd.Series:
    analyzer = SentimentIntensityAnalyzer()
    return texts.apply(
        lambda t: float(analyzer.polarity_scores(t)["compound"]) if t else 0.0
    )

In [0]:
df_silver = (
    df_cleaned.withColumn(
        "sentiment_score", fast_sentiment_udf(F.col("clean_text"))
    )
    .withColumn(
        "sentiment_label",
        F.when(F.col("sentiment_score") >= 0.05, "Positive")
        .when(F.col("sentiment_score") <= -0.05, "Negative")
        .otherwise("Neutral"),
    )
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "post_uri",
        "author_handle",
        "search_topic",
        "clean_text",
        "sentiment_score",
        "sentiment_label",
        "post_timestamp",
        "post_date",
        "post_hour",
        "ingestion_timestamp",
        "silver_processed_at",
    )
)

In [0]:
silver_table = "us_economy_db.silver_bsky_sentiment"

if not spark.catalog.tableExists(silver_table):
    (
        df_silver.write.format("delta")
        .partitionBy("post_date")
        .saveAsTable(silver_table)
    )
else:
    delta_target = DeltaTable.forName(spark, silver_table)
    (
        delta_target.alias("target")
        .merge(df_silver.alias("source"), "target.post_uri = source.post_uri")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

print(
    f"Data after transformation: {spark.table(silver_table).count()}"

)
display(spark.table(silver_table).limit(15))